In [0]:
# %pip install azure-eventhub
# %pip install requests

# Restart python after install
# dbutils.library.restartPython()

In [0]:
# %pip install 

In [0]:
# dbutils.library.restartPython()

In [0]:


import json
import time
import requests
from datetime import datetime, timezone
# from azure.eventhub import EventHubProducerClient, EventData

# ── CELL 3: Config ────────────────────────────────
CONNECTION_STRING = " "
EVENT_HUB_NAME    = "stock-prices"
API_KEY           = " 3KCA2GARAX4YHIOB."
STOCKS = ["MSFT", "AAPL", "GOOGL", "AMZN", "TSLA"]

# ── CELL 4: Functions ─────────────────────────────
def get_stock_data(ticker):
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "GLOBAL_QUOTE",
        "symbol":   ticker,
        "apikey":   API_KEY
    }
    response = requests.get(url, params=params)
    data     = response.json()
    quote    = data.get("Global Quote", {})

    if not quote:
        print(f"⚠️ No data for {ticker}")
        return None

    return {
        "ticker":        ticker,
        "open_price":    float(quote.get("02. open", 0)),
        "high_price":    float(quote.get("03. high", 0)),
        "low_price":     float(quote.get("04. low", 0)),
        "current_price": float(quote.get("05. price", 0)),
        "volume":        int(quote.get("06. volume", 0)),
        "close_price":   float(quote.get("08. previous close", 0)),
        "change":        float(quote.get("09. change", 0)),
        "change_pct":    quote.get("10. change percent", "0%").replace("%", ""),
        "timestamp":     datetime.now(timezone.utc).isoformat()
    }

def send_to_eventhub(stock_data):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=CONNECTION_STRING,
        eventhub_name=EVENT_HUB_NAME
    )
    with producer:
        batch = producer.create_batch()
        batch.add(EventData(json.dumps(stock_data)))
        producer.send_batch(batch)
        print(f"✅ Sent: {stock_data['ticker']} | "
              f"Price: ${stock_data['current_price']} | "
              f"Change: {stock_data['change_pct']}%")

# ── CELL 5: Run ONCE ──────────────────────────────
print("🚀 Stock Producer Started!")
print(f"📊 Tracking: {', '.join(STOCKS)}")
print("─" * 50)

for ticker in STOCKS:
    try:
        stock_data = get_stock_data(ticker)
        if stock_data:
            send_to_eventhub(stock_data)
            time.sleep(15) # ← 15 sec between stocks
                           #   to avoid API rate limit
    except Exception as e:
        print(f"❌ Error for {ticker}: {e}")

print("✅ All 5 stocks sent!")
print("✅ Producer completed!")


